# SuperSTAC — Many catalogs. One search.

Search Sentinel-2 item metadata across **Earth Search** and **Microsoft Planetary Computer**, inspect the combined response, map scene footprints, and export GeoJSON. No API key or GPU is needed for these public metadata searches.

**In Colab:** choose a standard Python CPU runtime, then run cells from top to bottom. You can upload this `.ipynb` using **File → Upload notebook**. In Jupyter, use a Python 3.9+ environment compatible with the current SuperSTAC wheel.

This notebook describes SuperSTAC's current alpha API. Public catalogs and released package versions can change, so results and counts are not fixed. Internet access is required for package installation, catalog requests, and map tiles.

[Python guide](https://spatialnode.com/superstac/docs/v0.3/python/overview) · [Source repository](https://github.com/Spatialnode/superstac)

## 1. Install and import

SuperSTAC provides the search engine. Pandas displays tables; Folium displays the map. The install cell targets the active notebook kernel.

If a compatible wheel is unavailable, follow the repository's [source installation guide](https://github.com/Spatialnode/superstac/blob/main/docs/content/docs/v0.3/start/installation.md). If you replace an already-imported version, restart the runtime before continuing. The version printed below helps when reporting an issue.

In [ ]:
%pip install --quiet "superstac>=0.3,<0.4" pandas folium

In [ ]:
import json
import platform
from importlib.metadata import version
from pathlib import Path
from zipfile import ZipFile

import folium
import pandas as pd
from IPython.display import FileLink, display
from superstac import Client

print(f"Python {platform.python_version()}")
print(f"SuperSTAC {version('superstac')}")

## 2. Choose the catalogs and query

The bounding box covers Luxembourg and its surroundings, in **[west, south, east, north] longitude/latitude degrees (EPSG:4326)**. Change it and the date interval to explore another area.

`limit=5` is a **per-catalog cap**, so two catalogs can contribute up to ten items before deduplication. It is not a global limit, a count of every matching scene, or a guarantee of the newest or least cloudy scenes. Deduplication compares item IDs; the same observation with different IDs can remain in the output.

For a short notebook session, each catalog disables its background health monitor. Startup health checks and collection discovery still run.

In [ ]:
CATALOGS = [
    {
        "id": "earth-search",
        "url": "https://earth-search.aws.element84.com/v1",
        "settings": {
            "health_check_strategy": "hourly",
            "healthy_status_code_range": [200, 299],
            "enable_background_health_monitor": False,
        },
    },
    {
        "id": "microsoft",
        "url": "https://planetarycomputer.microsoft.com/api/stac/v1",
        "settings": {
            "health_check_strategy": "hourly",
            "healthy_status_code_range": [200, 299],
            "enable_background_health_monitor": False,
        },
    },
]

QUERY = {
    "collections": ["sentinel-2-l2a"],
    "bbox": [6.0, 49.0, 7.0, 50.0],
    "datetime": "2024-01-01T00:00:00Z/2024-01-31T23:59:59Z",
    "limit": 5,
}

SETTINGS = {
    "max_concurrent_catalogs": 2,
    "per_catalog_timeout_seconds": 30,
    "max_retry_attempts": 2,
    "max_items_per_catalog": 5,
    "deduplicate_items": True,
    "unify_response": True,
}

print(json.dumps(QUERY, indent=2))

## 3. Discover collections and run one federated search

SuperSTAC starts the engine, discovers which catalogs advertise the collection, and searches the selected sources concurrently. Startup can take longer than the search itself; the per-catalog search timeout does not bound the entire startup process.

The `finally` block closes the client even if discovery or search raises an exception. Results are materialized before shutdown and remain available to the later cells. To try a different query, edit the configuration above and rerun this cell and the result cells below.

In [ ]:
# Clear previous results so a failed rerun cannot accidentally reuse them.
search = None
items = []
metadata = {}
geojson = {"type": "FeatureCollection", "features": []}

client = Client(catalogs=CATALOGS, settings=SETTINGS)
try:
    available = client.list_collections()
    selected = [entry for entry in available if entry["id"] in QUERY["collections"]]
    print("Discovery: catalogs advertising the requested collection")
    display(pd.DataFrame(selected, columns=["id", "catalogs"]))

    search = client.search(**QUERY)
    items = search.items()
    metadata = search.metadata
    geojson = search.to_geojson()
finally:
    client.shutdown()

print(f"Returned {search.matched()} items after aggregation.")

## 4. Check completeness before using the data

A response can contain useful items and catalog failures at the same time. A healthy/capability filter can also exclude a catalog before execution without recording a search failure. Compare the configured catalog count, discovery table, and metadata rather than assuming every source was searched.

`matched()` and `total_items` count returned items after aggregation, not all scenes available upstream. Python exposes ordinary item dictionaries and run-level metadata; it does not expose the Rust per-item `catalog_id` / `seen_in` provenance wrapper.

In [ ]:
if search is None:
    raise RuntimeError("Run the search cell successfully before inspecting results.")

print(json.dumps(metadata, indent=2))

if metadata.get("catalogs_queried", 0) < len(CATALOGS):
    print("Some configured catalogs were not selected. Check discovery and catalog health.")
for failure in metadata.get("failures", []):
    print(f"Catalog failure: {failure['catalog_id']}: {failure['reason']}")
if not items:
    print("No items returned. Check failures and unsupported collections before widening the date range or area.")

In [ ]:
rows = []
for item in items:
    properties = item.get("properties", {})
    rows.append({
        "id": item["id"],
        "collection": item.get("collection"),
        "datetime": properties.get("datetime") or properties.get("start_datetime"),
        "cloud_cover_percent": properties.get("eo:cloud_cover"),
        "asset_count": len(item.get("assets", {})),
    })

item_table = pd.DataFrame(rows, columns=[
    "id", "collection", "datetime", "cloud_cover_percent", "asset_count",
])
display(item_table)

## 5. Map the scene footprints

The green rectangle is the search area; the blue polygons are returned scene footprints. A footprint can extend beyond the query area. Hover over a scene to inspect its ID. This displays metadata geometry over OpenStreetMap tiles, not satellite image pixels.

In [ ]:
west, south, east, north = QUERY["bbox"]
scene_map = folium.Map(
    location=[(south + north) / 2, (west + east) / 2],
    tiles="OpenStreetMap",
    zoom_start=7,
)
folium.Rectangle(
    bounds=[[south, west], [north, east]],
    color="#285d46",
    weight=3,
    fill=False,
    tooltip="Search area",
).add_to(scene_map)

# Keep the display layer small; the exported GeoJSON retains the full STAC items.
footprints = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "geometry": item["geometry"],
            "properties": {"id": item["id"], "collection": item.get("collection", "")},
        }
        for item in items if item.get("geometry")
    ],
}
if footprints["features"]:
    folium.GeoJson(
        footprints,
        name="Scene footprints",
        style_function=lambda feature: {
            "color": "#2563eb", "weight": 2, "fillOpacity": 0.08,
        },
        tooltip=folium.GeoJsonTooltip(fields=["id", "collection"]),
    ).add_to(scene_map)
else:
    print("No scene geometry to draw; showing the search area only.")

scene_map.fit_bounds([[south, west], [north, east]])
display(scene_map)

## 6. Inspect available assets

A STAC item links to data assets. Listing them does not download imagery. Asset keys differ by provider; some URLs require provider-specific authentication or signing. This example does not sign Planetary Computer URLs or fetch image pixels.

In [ ]:
if items:
    first_item = items[0]
    print(f"Assets for {first_item['id']}")
    display(pd.DataFrame([
        {"key": key, "type": asset.get("type"), "roles": asset.get("roles", []), "href": asset.get("href")}
        for key, asset in first_item.get("assets", {}).items()
    ], columns=["key", "type", "roles", "href"]))
else:
    print("No returned item to inspect.")

## 7. Export and download

Save full STAC items as a GeoJSON FeatureCollection, plus the search metadata and query separately. GeoJSON alone does not include the run-level failure counts. The map HTML uses online tiles and browser libraries when opened.

The output directory and ZIP below are overwritten when this cell is rerun. In Colab, runtime files are temporary: download the ZIP to keep them.

In [ ]:
if search is None:
    raise RuntimeError("Run the search cell successfully before exporting.")

output_dir = Path("superstac-output")
output_dir.mkdir(exist_ok=True)
exports = {
    "items.geojson": geojson,
    "search-metadata.json": metadata,
    "query.json": {"catalogs": CATALOGS, "settings": SETTINGS, "query": QUERY},
}
for filename, value in exports.items():
    (output_dir / filename).write_text(json.dumps(value, indent=2), encoding="utf-8")
scene_map.save(str(output_dir / "footprints.html"))

archive = Path("superstac-results.zip")
with ZipFile(archive, "w") as bundle:
    for filename in [*exports, "footprints.html"]:
        bundle.write(output_dir / filename, arcname=filename)
print(f"Saved {len(items)} items and search diagnostics to {archive.resolve()}")

In [ ]:
# Download in Colab, or display a file link in a local Jupyter notebook.
try:
    from google.colab import files
except ImportError:
    display(FileLink(str(archive)))
else:
    files.download(str(archive))

## Where to go next

- Change the bounding box or month, then rerun from the configuration cell onward.
- Add another compatible STAC API to `CATALOGS` and inspect discovery before searching.
- Increase both `QUERY["limit"]` and `SETTINGS["max_items_per_catalog"]` to collect more items per catalog.
- Learn about [collection and asset aliases](https://spatialnode.com/superstac/docs/v0.3/guides/aliases), the [async client](https://spatialnode.com/superstac/docs/v0.3/python/async), and [current limitations](https://spatialnode.com/superstac/docs/v0.3/reference/status).

If a source is unavailable, inspect `metadata["failures"]`, then rerun the search cell to create a fresh client and repeat startup checks. A successful search with zero results is different from a failed search. No specific scene IDs, source ordering, or duplicate counts are guaranteed.